In [ ]:
import numpy as np
from keras.datasets import mnist
from matplotlib import pyplot as plt
import collections

In [ ]:
# Load the MNIST dataset
(Xtr, Ltr), (X_test, L_test) = mnist.load_data()
print(Xtr.shape, Ltr.shape)
print(X_test.shape, L_test.shape)

In [ ]:
#Task1.1: Data Preprocessing
def center(X):
    newX = X - np.mean(X, axis=0)
    return newX

def standardize(X):
    newX = center(X) / (np.std(X, axis=0) + 1e-8)   # tiny eps to be safe
    return newX

def decorrelate(X):
    newX = center(X)
    cov = newX.T.dot(newX) / float(newX.shape[0])
    eigVals, eigVecs = np.linalg.eig(cov)
    decorrelated = newX.dot(eigVecs)   # use centered data
    return decorrelated

def whiten(X):
    newX = center(X)
    cov = newX.T.dot(newX) / float(newX.shape[0])
    eigVals, eigVecs = np.linalg.eig(cov)
    decorrelated = newX.dot(eigVecs)   # use centered data
    whitened = decorrelated / np.sqrt(eigVals + 1e-5)
    return whitened



In [ ]:
import matplotlib.pyplot as plt

def plotDataAndCov(X, title=""):
    plt.figure(figsize=(4,4))
    plt.scatter(X[:,0], X[:,1], s=10)
    plt.title(title)
    plt.axis('equal')
    plt.grid(True)
    plt.show()

np.random.seed(1234)
c1 = np.random.normal(3, 1, 300)
c2 = c1 + np.random.normal(7, 5, 300)/2.
C = np.array([c1, c2]).T

plotDataAndCov(center(C), "Centered C")
plotDataAndCov(standardize(C), "Standardized C")
plotDataAndCov(decorrelate(C), "Decorrelated C")
plotDataAndCov(whiten(C), "Whitened C")


In [ ]:
# Task 1.2 1-nn classifier and its accuracy.
#Traing phase
num_sample=500
Tr_set=Xtr[:num_sample,:,:]
Ltr_set=Ltr[:num_sample]

Tr_set=Tr_set.reshape(num_sample,Tr_set.shape[1]*Tr_set.shape[2])

#Tr_set=Tr_set.reshape(num_sample,Tr_set.shape[1]*Tr_set.shape[2]).astype()
Tr_set.shape

In [ ]:
# Task 1.2 1-nn classifier and its accuracy.
def predict(X):
    num_test=X.shape[0]
    Lpred=np.zeros(num_test, dtype=Ltr_set.dtype)
    
    for i in range(num_test):
        #distances=np.sum(np.abs(Tr_set-X[i,:]),axis=1) # Uncomment for L1 
        distances=np.sqrt(np.sum((Tr_set-X[i,:])**2,axis=1)) #Task 1.3 to use L2 norm
        
        min_index= np.argmin(distances)
        Lpred[i]=Ltr_set[min_index]
    return Lpred

Test_images=X_test.reshape(X_test.shape[0],X_test.shape[1]* X_test.shape[2]) #  for Task 1.2 and
Test_images=Test_images.astype(np.float32) / 255.0  # Task 1.4 Bug FIX: Normalize pixel values to [0,1]
Labels_predicted=predict(Test_images)

print("Accuracy:", np.mean(Labels_predicted==L_test))

# Accuracy for 1-NN classifier method is around 0.2649 and its quite low beucase the training set
# is very small (500 samples) out of 60000 samples in the MNIST dataset.
# With limited training data, the 1-NN classifier cannot reliably represent the feature space

# The L2 norm performed worse than L1 norm and shows accuracy to drop by 7.5% form L1 norm
# This suggests that for  the MNIST pixel-space representation, the Manhattan distance (L1) is actually more effective 
# than Euclidean distance (L2) for nearest neighbor classification with a small training set.


# Bug FIX (Task 1.4): The issue was that pixel values were NOT normalized.
# MNIST images have pixel values in range [0, 255], but for distance-based classifiers,
# it's important to normalize to [0, 1] range to ensure proper distance calculations.
# This improves the accuracy to 0.8294! 
# The normalization ensures that distance metrics work properly and that the 1-NN classifier can effectively distinguish between different digit images.


In [ ]:
# Task 1.4: Hint - Checking distances manually vs function output
# Manual verification of distance calculations

# Manually compute L2 distance for first test image vs first few training images
print("\n=== Manual Distance Calculations ===")
test_img = Test_images[14]
for i in range(20):
    train_img = Tr_set[i]
    # Manual L2 distance calculation with Eculidean distance formula
    manual_distance = np.sqrt(np.sum((train_img.astype(np.float32) - test_img.astype(np.float32))**2))
    print(f"Sample {i}: L2 distance = {manual_distance:.2f}")

# we manually computed Euclidean distances between a test image and several training images using float32 arithmetic. 
# This confirmed that distances computed using uint8 values were incorrect due to wrap-around during subtraction. 
# Converting the data to float32 before distance computation fixed the issue.

In [ ]:
#Task 1.5
def predict(X, k=3):
    num_test = X.shape[0]
    Lpred = np.zeros(num_test, dtype=Ltr_set.dtype)

    for i in range(num_test):

        # --- Choose ONE distance metric ---
        distances = np.sum(np.abs(Tr_set - X[i, :]), axis=1)  # L1
        # distances = np.sqrt(np.sum((Tr_set - X[i, :])**2, axis=1))  # L2

        # --- k-NN voting ---
        k_indices = np.argsort(distances)[:k]     # indices of k closest training samples
        k_labels = Ltr_set[k_indices]             # labels of those k samples

        # Majority vote (most common label). Counts how many times each digit (0–9) appears in k_labels
        vote = np.bincount(k_labels, minlength=10).argmax()
        Lpred[i] = vote

    return Lpred

In [89]:
num_to_test = 100

Labels_predicted_k1 = predict(Test_images[:num_to_test], k=1)
acc1 = np.mean(Labels_predicted_k1 == L_test[:num_to_test])
print("1-NN accuracy:", acc1)

Labels_predicted_k3 = predict(Test_images[:num_to_test], k=3)
acc3 = np.mean(Labels_predicted_k3 == L_test[:num_to_test])
print("k-NN (k=3) accuracy:", acc3)

Labels_predicted_k7 = predict(Test_images[:num_to_test], k=7)
acc7 = np.mean(Labels_predicted_k7 == L_test[:num_to_test])
print("k-NN (k=7) accuracy:", acc7)


1-NN accuracy: 0.83
k-NN (k=3) accuracy: 0.79
k-NN (k=7) accuracy: 0.77
